<a href="https://colab.research.google.com/github/wikriwiki/dacon_1/blob/main/dacon_best.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DACON 구조물 안정성 추론 — 최고 성능 버전
**개선 사항 요약:**
- ✅ Train Augmentation (Flip, ColorJitter, RandomRotation, RandomErasing)
- ✅ Label Smoothing BCE Loss
- ✅ Early Stopping (patience=5)
- ✅ Warmup + CosineAnnealing 스케줄러
- ✅ Gradient Clipping
- ✅ Train+Dev 합산 재학습 (최종 제출 전)
- ✅ TTA (수평/수직 반전 앙상블)
- ✅ Classifier Head 단순화 (과적합 억제)

In [1]:
!pip install -q transformers accelerate timm peft

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q wandb

In [4]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: doinggyu (doinggyu-kwangwoon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm

# !pip install -q wandb
import wandb

# ── Config (Ablation Study 조절용 제어판) ──
CONFIG = {
    "seed": 42,
    "img_size": 224,
    "batch_size": 8,
    "epochs": 30,
    "patience": 5,
    "warmup_epochs": 2,
    "label_smooth": 0.05,
    "grad_clip": 1.0,
    "dropout": 0.3,

    # Ablation Toggles
    "use_cross_attention": True,  # True: Cross-Attention 사용, False: 단순 Concat
    "use_wandb": True,            # Wandb Logging (실행 시 로그인 화면 뜹니다)

    # LoRA 미사용 풀파인튜닝이므로 Backbone은 보수적인(매우 작은) LR 적용
    "lr_backbone": 5e-6,
    "lr_head": 1e-4,
    "model_name": "facebook/dinov2-base",
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])
print(f"Device: {DEVICE} / Config 초기화 완료")


Device: cuda / Config 초기화 완료


In [6]:
# ── Data Pipeline Module (원본 증강 복구) ──

class CenterPhysicsCrop:
    """물리 구조물 배경의 하늘/타일 노이즈를 먼저 잘라내는(Crop) 핵심 클래스"""
    def __init__(self, view: str):
        self.view = view

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        # 뷰(View)에 따라 찌그러짐 없이 타워가 위치한 중심 좌표 비율만 도려냄
        if self.view == "front":
            box = (int(0.25 * w), int(0.20 * h), int(0.75 * w), int(0.88 * h))
        else:
            box = (int(0.29 * w), int(0.29 * h), int(0.71 * w), int(0.71 * h))
        return img.crop(box)

def get_view_transforms(model_name, img_size):
    processor = AutoImageProcessor.from_pretrained(model_name)
    mean, std = processor.image_mean, processor.image_std

    # 각 시점(View)별 파이프라인 생성기
    def _train_tfm(view):
        return T.Compose([
            CenterPhysicsCrop(view=view),   # 1. 시점에 맞는 노이즈 크롭
            T.Resize((img_size, img_size)), # 2. 규격 맞춤

            # 3. 이하 원본의 극강 확률형 오버피팅 방지(Probabilistic) 증강들
            T.RandomApply([T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.20, hue=0.04)], p=0.8),
            T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 1.6))], p=0.35),
            T.RandomAdjustSharpness(sharpness_factor=0.8, p=0.2),
            T.RandomPerspective(distortion_scale=0.10, p=0.35),
            T.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.92, 1.08)),
            T.ToTensor(),
            T.RandomErasing(p=0.10, scale=(0.02, 0.08), ratio=(0.3, 3.3), value="random"),

            T.Normalize(mean=mean, std=std), # DINO 정규화
        ])

    def _val_tfm(view):
        return T.Compose([
            CenterPhysicsCrop(view=view),
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])

    # Front Train, Top Train, Front Val, Top Val 순서로 반환
    return _train_tfm("front"), _train_tfm("top"), _val_tfm("front"), _val_tfm("top")


class MultiViewDataset(Dataset):
    # 이제 단일 transform 대신 뷰(View)별 transform 2개를 따로 받습니다.
    def __init__(self, df, image_root, front_transform=None, top_transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.front_transform = front_transform
        self.top_transform = top_transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def _load_image(self, sample_id, view_name):
        path = os.path.join(self.image_root, str(sample_id), f"{view_name}.png")
        return Image.open(path).convert("RGB")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = row["id"]
        front_img = self._load_image(sample_id, "front")
        top_img   = self._load_image(sample_id, "top")

        # 각각의 시점(View) 파이프라인에 맞춰서 독립적으로 증강 처리
        if self.front_transform: front_img = self.front_transform(front_img)
        if self.top_transform:   top_img   = self.top_transform(top_img)

        item = {"id": sample_id, "front": front_img, "top": top_img}
        if not self.is_test:
            item["target"] = torch.tensor(row["target"], dtype=torch.float32)
        return item


def get_dataloaders(df_train, df_dev, train_dir, dev_dir, model_name, img_size, batch_size):
    tr_front, tr_top, val_front, val_top = get_view_transforms(model_name, img_size)

    train_ds = MultiViewDataset(df_train, train_dir, front_transform=tr_front, top_transform=tr_top)
    dev_ds   = MultiViewDataset(df_dev,   dev_dir,   front_transform=val_front, top_transform=val_top)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    dev_dl   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_dl, dev_dl


In [7]:
# ── Model Module ──
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),
                            (x.size(-2), x.size(-1))).pow(1.0 / self.p).flatten(1)

# 5. Cross-Attention 융합 모듈
class CrossAttentionFusion(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        # 정면(Front) 뷰가 기준이 되어 질문(Query)을 던지고 상단(Top) 뷰에서 답변(Key/Value)을 가져옴
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, front_feat, top_feat):
        # 1차원 벡터를 Attention 연산에 넣기 위해 임의의 순차 차원(길이 1) 부여 -> (B, 1, Dim)
        q = front_feat.unsqueeze(1)
        k = v = top_feat.unsqueeze(1)

        attn_out, _ = self.cross_attn(q, k, v)
        # 결과에 원래 Front를 더해주고(Residual) 정규화
        out = self.norm(q + attn_out).squeeze(1)
        return out


class AblationMultiViewClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()

        # 1. LoRA 제거 (순수 DINO 전체 파인튜닝)
        self.backbone = AutoModel.from_pretrained(config["model_name"])
        hidden_size = self.backbone.config.hidden_size

        self.gem = GeM(p=3.0)
        self.view_embed = nn.Parameter(torch.randn(2, hidden_size) * 0.02)

        # Config에 따라 Cross-Attention 사용 여부 결정 (Ablation 대비)
        self.use_cross_attention = config.get("use_cross_attention", False)
        if self.use_cross_attention:
            self.fusion_module = CrossAttentionFusion(embed_dim=hidden_size)
            classifier_in_dim = hidden_size * 2 # 원본 Front + Cross결과 Concat용
        else:
            classifier_in_dim = hidden_size * 2 # 기존 순수 단순 Concat

        self.classifier = nn.Sequential(
            nn.Linear(classifier_in_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(config["dropout"]),
            nn.Linear(512, 1)
        )

    def encode_image(self, x):
        outputs = self.backbone(pixel_values=x)
        patch_tokens = outputs.last_hidden_state[:, 1:]
        B, N, C = patch_tokens.shape
        H = W = int(N ** 0.5)
        patch_tokens = patch_tokens.transpose(1, 2).view(B, C, H, W)
        return self.gem(patch_tokens)

    def forward(self, front, top):
        f_front = self.encode_image(front)
        f_top   = self.encode_image(top)

        f_front = f_front + self.view_embed[0]
        f_top   = f_top   + self.view_embed[1]

        # ── 융합 제어판 ──
        if self.use_cross_attention:
            fused_cross = self.fusion_module(f_front, f_top)
            # Front 특징과 Cross Attention 해석 특징을 합쳐서 제출
            fused = torch.cat([f_front, fused_cross], dim=1)
        else:
            fused = torch.cat([f_front, f_top], dim=1)

        return self.classifier(fused).squeeze(1)


In [8]:
# ── Training & Evaluation Modules ──
class LabelSmoothingBCELoss(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, logits, targets):
        smooth_targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, smooth_targets)

def LOGLOSS(true, pred, eps=1e-15):
    pred = np.clip(pred, eps, 1 - eps)
    pred = pred / np.sum(pred, axis=1).reshape(-1, 1)
    loss = -np.sum(true * np.log(pred), axis=1)
    return np.mean(loss)

def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, total=len(loader), leave=False)
    for batch in pbar:
        front  = batch["front"].to(device)
        top    = batch["top"].to(device)
        target = batch["target"].to(device)

        optimizer.zero_grad()
        logits = model(front, top)
        loss   = criterion(logits, target)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item() * front.size(0)
        pbar.set_description(f"train loss: {loss.item():.4f}")

    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_targets, all_probs = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            front  = batch["front"].to(device)
            top    = batch["top"].to(device)
            target = batch["target"].to(device)
            logits = model(front, top)
            loss   = criterion(logits, target)
            probs  = torch.sigmoid(logits)

            total_loss += loss.item() * front.size(0)
            all_targets.extend(target.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except ValueError:
        auc = None

    probs_2d   = np.stack([1 - np.array(all_probs), np.array(all_probs)], axis=1)
    targets_2d = np.stack([1 - np.array(all_targets), np.array(all_targets)], axis=1)
    logloss    = LOGLOSS(targets_2d, probs_2d)

    return avg_loss, auc, logloss


In [10]:
# ── Main Execution ──
# (1) 경로 설정 (내 환경에 맞게 수정!)
DATA_ROOT = "/content/drive/MyDrive/dacon/"
TRAIN_CSV = f"{DATA_ROOT}/train.csv"
DEV_CSV   = f"{DATA_ROOT}/dev.csv"
TRAIN_DIR = f"{DATA_ROOT}/train"
DEV_DIR   = f"{DATA_ROOT}/dev"

label_map = {"stable": 0, "unstable": 1}
train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)
train_df["target"] = train_df["label"].map(label_map)
dev_df["target"]   = dev_df["label"].map(label_map)

# 3. Wandb 로깅 시작
if CONFIG["use_wandb"]:
    wandb.init(project="dacon_physics",
               name=f"ablation_CrossAttn_{CONFIG['use_cross_attention']}",
               config=CONFIG)

# 모델/데이터 준비
train_loader, dev_loader = get_dataloaders(
    train_df, dev_df, TRAIN_DIR, DEV_DIR,
    CONFIG["model_name"], CONFIG["img_size"], CONFIG["batch_size"]
)

# 1. LoRA 배제로 인한 옵티마이저 수정 (Backbone 풀파인튜닝 적용)
# 모델 전체(Backbone)의 체중을 파인튜닝해야 하므로 lr을 극도로 낮췄습니다(5e-6)
model = AblationMultiViewClassifier(CONFIG).to(DEVICE)
criterion = LabelSmoothingBCELoss(smoothing=CONFIG["label_smooth"])
backbone_params = [p for n, p in model.named_parameters() if "backbone" in n]
head_params     = [p for n, p in model.named_parameters() if "backbone" not in n]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": CONFIG["lr_backbone"]},
    {"params": head_params,     "lr": CONFIG["lr_head"]},
], weight_decay=1e-4)

# 스케줄러 세팅
def warmup_lambda(epoch):
    if epoch < CONFIG["warmup_epochs"]: return (epoch + 1) / CONFIG["warmup_epochs"]
    return 1.0

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lambda)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"] - CONFIG["warmup_epochs"], eta_min=1e-7)

# ── 학습 루프 (Wandb Logging 포함) ──
best_dev_logloss = 1e9
patience_cnt = 0

for epoch in range(CONFIG["epochs"]):
    print(f"\n[Epoch {epoch+1}/{CONFIG['epochs']}]")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, CONFIG["grad_clip"])
    dev_loss, dev_auc, dev_logloss = evaluate(model, dev_loader, criterion, DEVICE)

    if epoch < CONFIG["warmup_epochs"]:
        warmup_scheduler.step()
    else:
        cosine_scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Dev Loss: {dev_loss:.4f} | Dev AUC: {dev_auc:.4f} | Dev LogLoss: {dev_logloss:.4f}")

    # 3. Wandb 기록 업데이트
    if CONFIG["use_wandb"]:
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "dev_loss": dev_loss,
            "dev_auc": dev_auc,
            "dev_logloss": dev_logloss,
            "lr_backbone": optimizer.param_groups[0]['lr'],
            "lr_head": optimizer.param_groups[1]['lr']
        })

    # Early Stopping 판단
    if dev_logloss < best_dev_logloss:
        best_dev_logloss = dev_logloss
        patience_cnt = 0
        torch.save(model.state_dict(), "best_ablation_model.pt")
        print(f"  ⭐ Best Model Saved! (LogLoss: {best_dev_logloss:.4f})")
    else:
        patience_cnt += 1
        print(f"  Patience: {patience_cnt}/{CONFIG['patience']}")
        if patience_cnt >= CONFIG["patience"]:
            print("  🛑 Early Stopping Triggered!")
            break

# 로깅 종료
if CONFIG["use_wandb"]:
    wandb.finish()


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


[Epoch 1/30]


  0%|          | 0/125 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d58858d4680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process


Train Loss: 0.2386 | Dev Loss: 0.2019 | Dev AUC: 0.9984 | Dev LogLoss: 0.1374
  ⭐ Best Model Saved! (LogLoss: 0.1374)

[Epoch 2/30]


...

  0%|          | 0/125 [00:00<?, ?it/s]

Train Loss: 0.1449 | Dev Loss: 0.2542 | Dev AUC: 0.9988 | Dev LogLoss: 0.1740
  Patience: 4/5

[Epoch 17/30]


  0%|          | 0/125 [00:00<?, ?it/s]

Train Loss: 0.1381 | Dev Loss: 0.2079 | Dev AUC: 0.9944 | Dev LogLoss: 0.1224
  Patience: 5/5
  🛑 Early Stopping Triggered!


dev_auc,█▅▇▇███▇█▃▁█▃▇██▆
dev_logloss,▃█▃▂▃▅▁▂▂▇▅▁▅▂▂▄▃
dev_loss,▂█▃▂▃▄▁▂▂▇▅▁▅▂▂▄▃
epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
lr_backbone,█████▇▇▇▆▆▅▄▄▃▂▂▁
lr_head,█████▇▇▇▆▆▅▄▄▃▂▂▁
train_loss,█▄▃▂▂▂▂▁▁▃▃▁▃▂▁▂▁
dev_auc,0.99439
dev_logloss,0.12238
dev_loss,0.20789
epoch,17


In [11]:
# ── [Cell 6] 테스트 데이터 원클릭 추론 및 test.csv 저장 ──

def get_tta_transforms(model_name, img_size):
    processor = AutoImageProcessor.from_pretrained(model_name)
    mean, std = processor.image_mean, processor.image_std

    # 모델의 통찰력을 극대화하기 위한 3-Pass TTA 전략 (원본 1번 + 변형 2번 앙상블)
    def _tfm(view, tta_mode):
        base = [CenterPhysicsCrop(view=view), T.Resize((img_size, img_size))]
        if tta_mode == "color":
            base.append(T.ColorJitter(brightness=0.15, contrast=0.15))
        elif tta_mode == "blur":
            base.append(T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)))

        base.extend([T.ToTensor(), T.Normalize(mean=mean, std=std)])
        return T.Compose(base)

    tta_modes = ["original", "color", "blur"]
    tta_pipelines = []
    for mode in tta_modes:
        tta_pipelines.append((_tfm("front", mode), _tfm("top", mode))) # (front_tfm, top_tfm) 쌍으로 반환
    return tta_pipelines


def predict_test_tta(model, df, image_root, model_name, img_size, batch_size, device):
    model.eval()
    tta_pipelines = get_tta_transforms(model_name, img_size)
    all_probs = None

    print(f"총 {len(tta_pipelines)}가지의 TTA 앙상블로 추론을 시작합니다...")

    for i, (front_tfm, top_tfm) in enumerate(tta_pipelines):
        dataset = MultiViewDataset(df, image_root, front_transform=front_tfm, top_transform=top_tfm, is_test=True)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

        probs_run = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=f"TTA Pass {i+1}/{len(tta_pipelines)}"):
                front  = batch["front"].to(device)
                top    = batch["top"].to(device)

                logits = model(front, top)
                probs  = torch.sigmoid(logits).cpu().numpy().tolist()
                probs_run.extend(probs)

        probs_arr = np.array(probs_run)
        if all_probs is None:
            all_probs = probs_arr
        else:
            all_probs += probs_arr

    # TTA 3번의 예측값을 평균내어 가장 견고한 최종 확률 제출
    return all_probs / len(tta_pipelines)


# 1. 학습 과정에서 저장된 최고 성능(Best) 모델의 뇌(가중치) 불러오기
model.load_state_dict(torch.load("best_ablation_model.pt", map_location=DEVICE))

# 2. 제출용 포맷 파일(sample_submission) 및 Test 이미지 로드
SAMPLE_SUB_CSV = f"{DATA_ROOT}/sample_submission.csv"
TEST_DIR = f"{DATA_ROOT}/test"
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

# 3. 영끌 TTA 추론 실행
unstable_probs = predict_test_tta(
    model, sample_sub, TEST_DIR,
    CONFIG["model_name"], CONFIG["img_size"], CONFIG["batch_size"], DEVICE
)

# 4. 결과 정리 (규격에 맞게 확률 기입)
sample_sub["unstable_prob"] = unstable_probs
sample_sub["stable_prob"]   = 1.0 - unstable_probs

# 5. 지정하신 이름(test.csv)으로 현재 워크스페이스(혹은 루트 폴더)에 덮어쓰며 무조건 저장!
OUTPUT_PATH = f"{DATA_ROOT}/test.csv"
sample_sub.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"\n🎉 성공적으로 최종 추론이 오차 없이 완료되었습니다!")
print(f"▶ 엑셀 파일 저장 위치: {OUTPUT_PATH}\n")
print(sample_sub.head())


총 3가지의 TTA 앙상블로 추론을 시작합니다...


TTA Pass 1/3:   0%|          | 0/125 [00:00<?, ?it/s]

TTA Pass 2/3:   0%|          | 0/125 [00:00<?, ?it/s]

TTA Pass 3/3:   0%|          | 0/125 [00:00<?, ?it/s]


🎉 성공적으로 최종 추론이 오차 없이 완료되었습니다!
▶ 엑셀 파일 저장 위치: /content/drive/MyDrive/dacon//test.csv

          id  unstable_prob  stable_prob
0  TEST_0001       0.028539     0.971461
1  TEST_0002       0.990324     0.009676
2  TEST_0003       0.990952     0.009048
3  TEST_0004       0.988934     0.011066
4  TEST_0005       0.021709     0.978291
